# ISOM 839 · Session 8 — Nonlinear Optimization: Portfolios, Pricing & Curves

**Prescriptive Analytics: Modeling & Optimization · Suffolk University · Prof. Hasan Arslan**

Linearity has taken us far, but risk grows with the *square* of exposure and revenue bends as prices rise. Tonight: convexity (the line between easy and impossible), Markowitz's Nobel-winning model built from real market data in **CVXPY**, and the pricing curve that sets up Homework #7.

In [ ]:
%pip install -q cvxpy yfinance
import numpy as np, pandas as pd, cvxpy as cp
import matplotlib.pyplot as plt
print('cvxpy', cp.__version__, '· solvers:', cp.installed_solvers())

---
## Part 1 — Why linear isn't enough: a picture

Portfolio variance is **w'Σw** — a bowl. Revenue under a linear demand curve is **p(a − bp)** — an upside-down bowl. Both are *convex problems* (minimize a bowl, maximize an upside-down bowl): one smooth basin, so any local optimum is the global optimum and a solver can find it reliably. Non-convex problems are mountain ranges, and that is the whole difference between tonight and nightmares.

In [ ]:
p = np.linspace(0, 80, 200)
a, b = 1200, 15                                  # demand(p) = a - b*p
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].plot(p, p * (a - b * p)); ax[0].set_title('Revenue p·(a − b·p): concave, one peak'); ax[0].set_xlabel('price'); ax[0].axvline(a / (2 * b), ls='--', c='gray')
x = np.linspace(-3, 3, 200)
ax[1].plot(x, x**4 - 4 * x**2 + x); ax[1].set_title('A non-convex curve: two valleys'); ax[1].set_xlabel('x')
plt.tight_layout(); plt.show()

---
## Part 2 — Markowitz on real data (tonight's lab)

**Step 1 — get real data.** Two years of daily closes for five stocks via `yfinance`. (If the download is blocked where you are, the cell falls back to statistics saved from an earlier run so the lab still works.)

In [ ]:
tickers = ['AAPL', 'MSFT', 'JPM', 'XOM', 'JNJ']
try:
    import yfinance as yf
    prices = yf.download(tickers, period='2y', auto_adjust=True, progress=False)['Close'][tickers]
    rets = prices.pct_change().dropna()
    mu = rets.mean().values * 252                       # annualized expected return
    Sigma = rets.cov().values * 252                     # annualized covariance
    print(f'{len(rets)} trading days downloaded')
except Exception as e:
    print('Download unavailable — using saved statistics. Reason:', type(e).__name__)
    mu = np.array([0.22, 0.25, 0.18, 0.06, 0.04])
    vols = np.array([0.26, 0.24, 0.22, 0.24, 0.15])
    corr = np.array([[1.00, 0.65, 0.35, 0.15, 0.25],
                     [0.65, 1.00, 0.40, 0.15, 0.25],
                     [0.35, 0.40, 1.00, 0.30, 0.30],
                     [0.15, 0.15, 0.30, 1.00, 0.20],
                     [0.25, 0.25, 0.30, 0.20, 1.00]])
    Sigma = np.outer(vols, vols) * corr
n = len(tickers)
print(pd.DataFrame({'exp. return': mu, 'volatility': np.sqrt(np.diag(Sigma))}, index=tickers).round(3))
print('\nCorrelations:'); print(pd.DataFrame(Sigma / np.outer(np.sqrt(np.diag(Sigma)), np.sqrt(np.diag(Sigma))), index=tickers, columns=tickers).round(2))

**Step 2 — read the statistics like an investor.** Which pairs move together most, and which are nearly uncorrelated — or negative? Those low numbers are diversification you get for free, and that correlation table is where the portfolio's value comes from. (Real data: the numbers change every time you run this.)

**Step 3 — formulate and sweep.** Minimize variance subject to a target return, weights summing to one, no shorting. CVXPY checks that `quad_form` is convex before it lets you solve — if you ever write a non-convex objective by accident, it refuses, which is a feature.

In [ ]:
w = cp.Variable(n)
target = cp.Parameter(nonneg=True)
problem = cp.Problem(cp.Minimize(cp.quad_form(w, Sigma)), [mu @ w >= target, cp.sum(w) == 1, w >= 0])

frontier, weights = [], []
for r in np.linspace(mu.min(), mu.max(), 30):
    target.value = r
    problem.solve()
    if problem.status == 'optimal':
        frontier.append((np.sqrt(problem.value), r)); weights.append(w.value.copy())
risk, ret = zip(*frontier)
i_min = int(np.argmin(risk))
print(f'Minimum-variance portfolio: volatility {risk[i_min]:.1%}, return {ret[i_min]:.1%}')
print('  weights:', {t: round(float(x), 2) for t, x in zip(tickers, weights[i_min])})

In [ ]:
plt.figure(figsize=(7, 4.2))
plt.plot(risk, ret, lw=2, label='efficient frontier')
for i, t in enumerate(tickers):
    plt.scatter(np.sqrt(Sigma[i, i]), mu[i]); plt.annotate(t, (np.sqrt(Sigma[i, i]), mu[i]), xytext=(5, 3), textcoords='offset points')
plt.xlabel('volatility (annualized)'); plt.ylabel('expected return (annualized)'); plt.legend(); plt.grid(alpha=.3); plt.show()

**Step 4 — interpret.** Every single stock sits *below or to the right of* the curve. The vertical gap between a stock and the frontier at the same volatility is the measurable value of diversification — the most famous chart in finance, built by your own code in twenty lines.

### 💬 Discussion
Look at the minimum-variance weights. Did the optimizer concentrate in the "boring" stocks? Now imagine you manage $10B: which real constraints — sector caps, turnover limits, liquidity, ESG — would you add before trusting these weights, and which of them are *linear*? (Most are. The nonlinearity is all in the objective.)

---
## Part 3 — Pricing curves: the setup for Homework #7, Option B

A product's demand falls linearly with price: **demand(p) = a − b·p**. Revenue is **R(p) = p·(a − b·p)** — a downward parabola, so the optimal price is where the slope is zero: **p\* = a / 2b**. Here is the same thing in CVXPY (note `cp.square` so the objective is recognizably concave):

In [ ]:
a, b = 1200, 15
price = cp.Variable(nonneg=True)
rev = cp.Problem(cp.Maximize(a * price - b * cp.square(price)), [price <= a / b])
rev.solve()
print(f'Revenue-maximizing price: ${price.value:.2f}  (analytic a/2b = ${a / (2 * b):.2f})  revenue ${rev.value:,.0f}')

---
# Homework #7 · Option B — Revenue-Maximizing Pricing 💲

**Pick one of the two Homework #7 options on Canvas: this one, or the inventory MDP in the Session 10 notebook. Same points, same rubric.** Due date on Canvas.

Your product's demand curve, estimated from two years of sales: **a = 1,200, b = 15** (units per week, price in dollars). Unit cost **c = $12**.

1. **Revenue.** Formulate R(p) = p·(a − b·p). Solve for the optimal price **analytically and with CVXPY**, and confirm they agree.
2. **Profit.** Add the unit cost and maximize **(p − c)·(a − b·p)** instead. Solve again, both ways. Which direction did the price move, and why does cost *raise* the optimal price?
3. **Plot** the revenue curve and the profit curve on the same axes with both optima marked.
4. **One paragraph for a manager:** how does the optimal price change as **b** (price sensitivity) rises — try b = 10, 15, 20 — and what does that mean for how you set prices in a market that is becoming more competitive?

In [ ]:
a, b, c = 1200, 15, 12

# TODO 1: revenue-maximizing price, analytic and CVXPY

# TODO 2: profit-maximizing price, analytic and CVXPY  (hint: d/dp of (p-c)(a-bp) = 0)

# TODO 3: plot revenue and profit against price with both optima marked

# TODO 4: sweep b in [10, 15, 20] and print p* for each; write the paragraph in a markdown cell below

---
### Submission checklist (Option B)
- [ ] Both optima computed two ways, and they agree
- [ ] A sentence on why the profit-maximizing price is above the revenue-maximizing one
- [ ] The plot with both optima marked
- [ ] The manager paragraph — pasted into the Canvas text box, starting with "Option B"

**Next week — Session 9:** Markov chains, and the midterm case is due — briefings in class. 📈